In [0]:
# %pip install "loguru==0.7.3" --quiet

In [0]:
%restart_python


In [0]:
from pathlib import Path
from loguru import logger
from common.table_types import ProjectView
from common.table_types import ProjectTable
from common.table_defns.project._corpus_features import CorpusFeaturesTable
from common.table_defns.project import (
    create_mrns_table,
    create_cohort_table,
    create_corpus_features_table,
    create_note_types_table,
    get_corpus_definition_view,
)
from datetime import datetime

PROJECT_ROOT=Path("/Workspace/Users/dax.westerman@vumc.org/tedla-hypertension-project/v0.0.2")
LOGS_FOLDER = PROJECT_ROOT / "logs"
DEDUPED_FLAG_PATH = LOGS_FOLDER / "deduped_flag.txt"
SOURCE_MRN_CROSSWALK_FILE = "raw_data/patient_mrns/0.0.2_p6290_xwalk_mrn_deid.csv"

NOTE_DATE_LOWER_BOUND=datetime(2018, 1, 1)


# Added logger for the notebook
logger.remove()
logger.add(lambda msg: print(msg, end=""))
logger.add(LOGS_FOLDER / "error.logs", level="ERROR")
logger.add(LOGS_FOLDER / "processing.log", level="DEBUG")

3

In [0]:
#####################################################################################
# Build MRNs Table
#
# Takes the raw CSV file containing the MRN crosswalk and the cleaned
# patient cohort parquet file to create the MRNs table in the project.
#####################################################################################

from pathlib import Path
from loguru import logger
from common.table_types import ProjectView
from common.table_types import ProjectTable
from common.table_defns.project._corpus_features import CorpusFeaturesTable
from common.table_defns.project import (
    create_mrns_table,
    create_cohort_table,
    create_corpus_features_table,
    create_note_types_table,
    get_corpus_definition_view,
)

from common.files import CSVFileWrapper, ParquetFileWrapper
from common.table_types import RDTable
rd_note_table = RDTable("note")
rd_visit_occurrence_table = RDTable("visit_occurrence")
rd_person_table = RDTable("person")

mrns_table: ProjectTable

mrn_crosswalk_csv_file = CSVFileWrapper(
    name="patient_mrn_crosswalk",
    workspace_path=PROJECT_ROOT / SOURCE_MRN_CROSSWALK_FILE,
    dbfs_file_name="patient_cohort",
    table_name="mrns",
)

mrns_table: ProjectTable = create_mrns_table(
    mrn_crosswalk_csv_file
)


2025-11-12 23:26:07.660 | INFO     | common.table_types:create:64 - default.mrns table already exists.


In [0]:
# Build Cohort Table
patient_cohort_parquet_file = ParquetFileWrapper(
    name="patient_cohort", workspace_path=PROJECT_ROOT / "cleaned_data"
)
cohort_table: ProjectView = create_cohort_table(mrns_table, rd_person_table)
# patient_cohort_parquet_file.move_to("dbfs")
# cohort_table.create()

2025-11-12 23:26:08.188 | INFO     | common.table_types:create:64 - default.cohort table already exists.


In [0]:
from common.table_defns.project import create_corpus_features_table
# Build Corpus Features Table
corpus_features_source: CorpusFeaturesTable = create_corpus_features_table(cohort_table, rd_note_table, rd_visit_occurrence_table, NOTE_DATE_LOWER_BOUND)


2025-11-12 23:26:12.986 | INFO     | common.table_types:create:71 - Building default.vw_corpus_definition table...
2025-11-12 23:26:12.987 | INFO     | common.table_types:create:73 - Executing query

CREATE VIEW IF NOT EXISTS default.vw_corpus_definition AS 
 
            SELECT    n.note_id
                    , n.note_type_concept_id
                    , n.note_title
                    , n.note_source_value
                    , n.x_source
                    , v.visit_concept_id
                    , c.concept_name as discharge_to_concept_name
                    , p.specialty_concept_id as associated_provider_specialty_id
            FROM victr_rd.rd_omop_prod.note n
            INNER JOIN default.cohort c
                ON n.person_id = c.person_id
            INNER JOIN victr_rd.rd_omop_prod.visit_occurrence v
                on n.visit_occurrence_id = v.visit_occurrence_id
            inner join `victr_rd`.`rd_omop_prod`.`concept` `c`
                on c.concept_id = v.disch

In [0]:
# Build Note Types Table
note_types_table: ProjectTable = create_note_types_table()


2025-11-12 23:28:51.327 | INFO     | common.table_types:create:71 - Building default.note_types table...
2025-11-12 23:28:51.327 | INFO     | common.table_types:create:73 - Executing query

CREATE TABLE IF NOT EXISTS default.note_types AS 

        select 1 as note_type_id, 'notes_admissions' as note_type
        union all
        select 2 as note_type_id, 'notes_communication_encounter' as note_type
        union all
        select 3 as note_type_id, 'notes_discharge_summary' as note_type
        union all
        select 4 as note_type_id, 'notes_ecg_impression' as note_type
        union all
        select 5 as note_type_id, 'notes_emergency_department' as note_type
        union all
        select 6 as note_type_id, 'notes_inpatient' as note_type
        union all
        select 7 as note_type_id, 'notes_outpatient' as note_type
        union all
        select 8 as note_type_id, 'notes_problem_lists' as note_type
        
2025-11-12 23:28:51.467 | INFO     | common.table_types:crea

In [0]:
# # Build Corpus Definition View
# corpus_definition_view: ProjectView = get_corpus_definition_view(cohort_table, rd_note_table, rd_visit_occurrence_table)
# corpus_definition_view.create()

In [0]:
# ......................................................................
# Static definitions of note groups
# ......................................................................

from .common.table_definitions.project.note_groups import (
    AdmissionNotesTable,
    CommunicationEncounterNotesTable,
    DischargeSummaryNotesTable,
    ECGImpressionNotesTable,
    EmergencyDepartmentNotesTable,
    InpatientNotesTable,
    OutpatientNotesTable,
    ProblemListNotesTable,
)

admission_notes_table: AdmissionNotesTable = AdmissionNotesTable(corpus_features_source).create()

In [0]:
communication_encounter_notes_table: CommunicationEncounterNotesTable = CommunicationEncounterNotesTable(
    corpus_features_source
).create()

In [0]:
discharge_summary_notes_table: DischargeSummaryNotesTable = DischargeSummaryNotesTable(corpus_features_source).create()

In [0]:
ecg_impression_notes_table: ECGImpressionNotesTable = ECGImpressionNotesTable(corpus_features_source).create()

In [0]:
emergency_department_notes_table: EmergencyDepartmentNotesTable = EmergencyDepartmentNotesTable(corpus_features_source).create()

In [0]:
inpatient_notes_table: InpatientNotesTable = InpatientNotesTable(corpus_features_source).create()

In [0]:
outpatient_notes_table: OutpatientNotesTable = OutpatientNotesTable(corpus_features_source).create()

In [0]:
problem_list_notes_table: ProblemListNotesTable = ProblemListNotesTable(corpus_features_source).create()